<a href="https://colab.research.google.com/github/nickplas/Intro_to_ML_25-26/blob/main/notebooks/Lab-7.KNNGaussianNaiveBayesTrees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification with KNN, Trees and Gaussian Naive Bayes

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler

Load and split the data from the Unsupervise Learning Dataset (Lab 5, Dry Bean Dataset):

In [18]:
FFILE = './Dry_Bean_Dataset.xlsx'
if os.path.isfile(FFILE): 
    print("File already exists")
    if os.access(FFILE, os.R_OK):
        print ("File is readable")
    else:
        print ("File is not readable, removing it and downloading again")
        !rm FFILE
        !wget "https://raw.github.com/alexdepremia/ML_IADA_UTs/main/Lab5/Dry_Bean_Dataset.xlsx"
else:
    print("Either the file is missing or not readable, download it")
    !wget "https://raw.github.com/alexdepremia/ML_IADA_UTs/main/Lab5/Dry_Bean_Dataset.xlsx"

File already exists
File is readable


In [19]:
# Load the data
data = pd.read_excel('./Dry_Bean_Dataset.xlsx')
data

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13606,42097,759.696,288.721612,185.944705,1.552728,0.765002,42508,231.515799,0.714574,0.990331,0.916603,0.801865,0.006858,0.001749,0.642988,0.998385,DERMASON
13607,42101,757.499,281.576392,190.713136,1.476439,0.735702,42494,231.526798,0.799943,0.990752,0.922015,0.822252,0.006688,0.001886,0.676099,0.998219,DERMASON
13608,42139,759.321,281.539928,191.187979,1.472582,0.734065,42569,231.631261,0.729932,0.989899,0.918424,0.822730,0.006681,0.001888,0.676884,0.996767,DERMASON
13609,42147,763.779,283.382636,190.275731,1.489326,0.741055,42667,231.653248,0.705389,0.987813,0.907906,0.817457,0.006724,0.001852,0.668237,0.995222,DERMASON


Divide features and label. Split the data in train and test set and **after that** normalize them:

In [20]:
data = data.sample(frac =1, random_state=0).reset_index(drop=True)

data.head()


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,37277,710.193,264.789840,179.808422,1.472622,0.734082,37684,217.859015,0.802692,0.989200,0.928748,0.822762,0.007103,0.002008,0.676937,0.996873,DERMASON
1,28942,638.821,239.861192,154.004371,1.557496,0.766658,29368,191.963796,0.786126,0.985494,0.891210,0.800312,0.008288,0.002097,0.640499,0.997575,DERMASON
2,38290,719.888,270.446510,180.508066,1.498252,0.744659,38605,220.799326,0.759903,0.991840,0.928465,0.816425,0.007063,0.001936,0.666550,0.998660,DERMASON
3,37641,742.538,284.313737,169.740814,1.674987,0.802227,38112,218.920099,0.744187,0.987642,0.857894,0.769995,0.007553,0.001638,0.592892,0.993087,SIRA
4,50172,828.968,316.453571,202.268818,1.564520,0.769062,50547,252.746858,0.688240,0.992581,0.917478,0.798685,0.006307,0.001583,0.637898,0.998005,SEKER


In [21]:
data.shape

(13611, 17)

In [22]:
data['Class'].unique()

array(['DERMASON', 'SIRA', 'SEKER', 'CALI', 'BOMBAY', 'HOROZ', 'BARBUNYA'],
      dtype=object)

In [23]:
train_data = data.iloc[:10000,:] # 70% dati per train
test_data = data.iloc[10000:,:] # 30% dati per test

In [24]:
print(train_data.shape)
print(test_data.shape)


(10000, 17)
(3611, 17)


In [25]:
# normalize train and test dataset
from sklearn import preprocessing

# Extract the class labels from the training dataset
label_train = train_data['Class']
# Remove the class labels from the training dataset
train_data = train_data.drop('Class', axis=1)
# Save the column names for later use
columns_name = train_data.columns

# Initialize a StandardScaler and fit it to the training data
train_scaler = preprocessing.StandardScaler().fit(train_data)
# Transform the training data using the scaler
train_data = train_scaler.transform(train_data)
# Create a DataFrame with the scaled training data and restore column names
train_data = pd.DataFrame(train_data, columns=columns_name)
# Add the class labels back to the scaled training dataset
train_data['Class'] = label_train

# Extract the class labels from the test dataset
label_test = test_data['Class']
# Remove the class labels from the test dataset
test_data = test_data.drop('Class', axis=1)
# Initialize a StandardScaler and fit it to the test data
test_scaler = preprocessing.StandardScaler().fit(test_data)
# Transform the test data using the scaler
test_data = test_scaler.transform(test_data)
# Create a DataFrame with the scaled test data and restore column names
test_data = pd.DataFrame(test_data, columns=columns_name)
# Add the class labels back to the scaled test dataset
test_data['Class'] = label_test.values # .values added to prevent nans to appear

In [26]:
test_data.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,-0.491627,-0.569589,-0.656984,-0.333932,-0.706709,-0.497404,-0.489871,-0.525182,-0.717967,-0.247282,0.533742,0.669072,0.220508,0.646010,0.644996,0.548373,DERMASON
1,-0.178177,-0.303014,-0.508053,0.407000,-1.332487,-1.585654,-0.187714,-0.105056,0.770204,1.194325,1.515348,1.491940,-0.692110,0.901487,1.533549,0.899191,SEKER
2,-0.825667,-1.017373,-0.954579,-1.055949,-0.179213,0.103660,-0.826306,-1.032558,-0.899978,0.385969,0.384324,0.090319,1.469507,0.697635,0.051499,0.837196,DERMASON
3,-0.496063,-0.650030,-0.775585,-0.186687,-1.113788,-1.140660,-0.499874,-0.531467,-0.096816,0.671346,1.227555,1.189937,0.008414,1.071414,1.201339,0.846310,SEKER
4,-0.439476,-0.435341,-0.383451,-0.494706,0.050079,0.309489,-0.433755,-0.452065,-1.020644,-0.881375,-0.044013,-0.196255,0.506766,-0.036555,-0.232759,-1.134762,SIRA


In [27]:
train_data.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,-0.534137,-0.673988,-0.642784,-0.494897,-0.448854,-0.185185,-0.536571,-0.590932,1.083156,0.441530,0.932328,0.372606,0.473404,0.490160,0.338721,0.416048,DERMASON
1,-0.819126,-1.008115,-0.934618,-1.070089,-0.104138,0.170622,-0.816632,-1.029793,0.746572,-0.348554,0.302277,0.007584,1.526541,0.640814,-0.030790,0.576165,DERMASON
2,-0.499500,-0.628601,-0.576563,-0.479301,-0.344759,-0.069652,-0.505554,-0.541101,0.213791,1.004632,0.927566,0.269572,0.437664,0.368508,0.233385,0.823378,DERMASON
3,-0.521691,-0.522565,-0.414222,-0.719312,0.373055,0.559127,-0.522157,-0.572949,-0.105517,0.109317,-0.256904,-0.485355,0.873546,-0.133675,-0.513569,-0.447017,SIRA
4,-0.093232,-0.117944,-0.037969,0.005762,-0.075610,0.196889,-0.103380,0.000331,-1.242245,1.162580,0.743167,-0.018863,-0.234347,-0.225790,-0.057166,0.674089,SEKER


In [28]:
test_data['Class'].unique()

array(['DERMASON', 'SEKER', 'SIRA', 'BARBUNYA', 'HOROZ', 'BOMBAY', 'CALI'],
      dtype=object)

**Before feeding the data into the following algorithms, try to perform PCA, varying the number of PCs, and check what changes**

PCA DA FARE A CASA ( FALLO APPENA PUOI )

## K-Nearest Neighbors Classification 

Implement the KNN algorithm for classification.

In [29]:
from scipy.spatial.distance import euclidean, minkowski
from tqdm import tqdm

def distance(point_one, point_two, dist, p):
    """
    Calculate the Euclidean distance between two points.

    Parameters
    ----------
    point_one : array-like
        Coordinates of the first point.
    point_two : array-like
        Coordinates of the second point.
    dist: str
        Allow to choose between Euclidean or Minkowski distance.
    p: int
        Order of the norm, only used with Minkowski distance.

    Returns
    -------
    float
        Euclidean or Minkowski distance between the two points.
    """
    if dist == 'euclidean':
        return euclidean(point_one, point_two)
    else:
        return minkowski(point_one, point_two, p=p)


def get_neighbors(train_set, test_point, label_col, n_neighbors, dist, p):
    """
    Get the nearest neighbors of a test point in the training set.

    Parameters
    ----------
    train_set : array-like
        The training set containing data points.
    test_point : array-like
        The test point for which neighbors are to be found.
    label_col : array-like
        The labels corresponding to the training set.
    n_neighbors : int
        The number of neighbors to retrieve.
    dist: str
        Allow to choose between Euclidean or Minkowski distance.
    p: int
        Order of the norm, only used with Minkowski distance.

    Returns
    -------
    ordered_train : array-like
        The nearest neighbors in the training set.
    ordered_label : array-like
        The corresponding labels of the nearest neighbors.
    """
    # Calculate distances between the test point and all points in the training set
    dist = np.array([distance(train_point, test_point, dist, p) for train_point in train_set])
    # Get indices that would sort the distances in ascending order
    idx_dist = dist.argsort()
    # Order the training set and labels based on the sorted distances
    ordered_train = train_set[idx_dist, :]
    ordered_label = label_col[idx_dist]
    # Return the top n_neighbors neighbors and their labels
    return ordered_train[:n_neighbors], ordered_label[:n_neighbors]

def predict(train_set, test_point, labels, n_neighbors, dist, p):
    """
    Predict the label of a test point using k-nearest neighbors.

    Parameters
    ----------
    train_set : array-like
        The training set containing data points.
    test_point : array-like
        The test point for which the label is to be predicted.
    labels : array-like
        The labels corresponding to the training set.
    n_neighbors : int
        The number of neighbors to consider for the prediction.
    dist: str
        Allow to choose between Euclidean or Minkowski distance.
    p: int
        Order of the norm, only used with Minkowski distance.

    Returns
    -------
    predicted_label : array-like
        The predicted label for the test point.
    """
    # Get the nearest neighbors and their labels
    neigh, neigh_label = get_neighbors(train_set, test_point, labels, n_neighbors, dist, p)
    # Count occurrences of each label among the neighbors
    values, counts = np.unique(neigh_label, return_counts=True)
    # Find the label with the highest count (majority class)
    idx = np.argmax(counts)
    # Return the predicted label
    return values[idx]

def evaluate(train_set, test_set, label, n_neighbors=2, dist='Euclidean', p=2):
    """
    Evaluate the accuracy of k-nearest neighbors algorithm on a test set.

    Parameters
    ----------
    train_set : DataFrame
        The training dataset.
    test_set : DataFrame
        The test dataset.
    label : str
        The name of the column representing the class labels.
    n_neighbors : int, optional
        The number of neighbors to consider for the prediction. Default is 2.
    dist: str
        Allow to choose between Euclidean or Minkowski distance.
    p: int
        Order of the norm, only used with Minkowski distance.

    Returns
    -------
    accuracy : float
        The accuracy of the k-nearest neighbors algorithm on the test set.
    """
    # Initialize counters for correct and incorrect predictions
    correct_predict = 0
    wrong_predict = 0
    # Extract labels and features from the training and test sets
    
    train_labels = train_set[label].values
    train_set = train_set.drop(label, axis=1)
    
    test_labels = test_set[label].values
    test_set = test_set.drop(label, axis=1)
    # Iterate through each row in the test dataset
    for index in tqdm(range(len(test_set.index)), desc="Evaluating KNN"):
        # Predict the class label for the current test row
        result = predict(train_set.values, test_set.iloc[index].values, train_labels, n_neighbors, dist, p)
        # Check if the predicted value matches the actual value
        if result == test_labels[index]:
            # Increase the correct prediction count
            correct_predict += 1
        else:
            # Increase the incorrect prediction count
            wrong_predict += 1

    # Calculate and return the accuracy
    accuracy = correct_predict / (correct_predict + wrong_predict)
    return accuracy


In [30]:
knn_accuracy = evaluate(train_data, test_data, 'Class', n_neighbors=5)

Evaluating KNN: 100%|██████████| 3611/3611 [03:06<00:00, 19.34it/s]


In [31]:
knn_accuracy

0.9243976737745777

## Decision Trees with Numerical Features 

Modify the implementation of decision trees to account for numerical input features.

In [32]:
# compute H(S)
def entropy(train_data, label, class_list):
    """
    Calculate the entropy of a dataset.

    Parameters
    ----------
    train_data : DataFrame
        The training dataset.
    label : str
        The name of the column representing the class labels.
    class_list : list of str
        List of possible values of the class labels.

    Returns
    -------
    total_entr : float
        The entropy of the dataset.
    """
    # Get the total number of instances in the dataset
    total_row = train_data.shape[0]
    # Initialize the total entropy variable
    total_entr = 0

    # Iterate through each possible class in the label
    for c in class_list:
        # Count the number of points belonging to the current class
        total_class_count = train_data[train_data[label] == c].shape[0]

        # Check if there are instances of the class to avoid numerical errors
        if total_class_count > 0:
            # Calculate the entropy of the current class
            total_class_entr = - (total_class_count / total_row) * np.log2(total_class_count / total_row)
            # Add the entropy of the current class to the total entropy of the dataset
            total_entr += total_class_entr

    # Return the calculated total entropy of the dataset
    return total_entr

In [33]:
# compute H(S_j)
def feature_entropy(left_data, right_data, label, class_list):
    """
    Calculate the conditional entropy of a dataset split by a specific feature.

    Parameters
    ----------
    left_data : DataFrame
        Subset of the dataset where the feature has a specific value.
    right_data : DataFrame
        Subset of the dataset where the feature has another value.
    label : str
        The name of the column representing the class labels.
    class_list : list of str
        List of possible values of the class labels.

    Returns
    -------
    ent : float
        The conditional entropy of the dataset split by the feature.
    """
    # Get the total number of points considered after the split
    row_count = left_data.shape[0] + right_data.shape[0]

    # Calculate the probabilities of the left and right subsets
    p_left = left_data.shape[0] / row_count
    p_right = right_data.shape[0] / row_count

    # Calculate the conditional entropy using the weighted average of entropies for left and right subsets
    ent = p_left * entropy(left_data, label, class_list) + p_right * entropy(right_data, label, class_list)

    # Return the calculated conditional entropy
    return ent

In [34]:
def split(feature_column, threshold):
    """
    Split the indices of data points based on a feature and a threshold.

    Parameters
    ----------
    feature_column : array-like
        The values of the feature for each data point.
    threshold : float
        The threshold value for splitting the data points.

    Returns
    -------
    left_rows : array-like
        Indices of data points where the feature value is less than or equal to the threshold.
    right_rows : array-like
        Indices of data points where the feature value is greater than the threshold.
    """
    # Find the indices of data points where the feature value is less than or equal to the threshold
    left_rows = np.argwhere(feature_column <= threshold).flatten()
    # Find the indices of data points where the feature value is greater than the threshold
    right_rows = np.argwhere(feature_column > threshold).flatten()

    # Return the indices for left and right subsets
    return left_rows, right_rows

## Gaussian Naive Bayes 
Modufy the implemntation of naive Bayes to accout for numerical input features. The likelihood of each class ($p(data|class)$) is assumed to be a Gaussian $\frac{1}{\sqrt(\sigma^2 2 \pi)} \exp (\frac{1}{2} \frac{(x-\mu)}{\sigma^2})$, where $\mu, \sigma^2$ are the mean and the variance for each class;